### Import modules and utility code

In [ ]:
import os
import numpy as np
from score_config import ScoreConfig
from score_config_utils import ureg
from pint import Quantity
from typing import List, Dict, Any, Tuple

print(os.getcwd())

### Create a configuration file for SCORE following the NXmicrostructure_score_config.nxdl.xml contributed application definition

<div class="alert alert-block alert-info">Modify those values in this populated template that match your case.</div>

In [ ]:
MY_MATERIAL: List[Tuple[str, Quantity]] = [
    ("shear_modulus", ureg.Quantity(np.float64(9.5e9), ureg.pascal)),
    ("burgers_vector", ureg.Quantity(np.float64(0.286), ureg.nanometer)),
    (
        "melting_temperature",
        ureg.Quantity(np.float64(800.0), ureg.degC).to(ureg.kelvin),
    ),
]

MY_USER: List[str] = [
    "Vitesh Shah",
    "Markus Kühbach",
    "Sebastian Samberger",
    "Georg Falkinger",
    "Franz Roters",
]

MY_INIT_DAMASK: Dict[str, str] = {
    "file_path": "../../examples/damask/Tphase_Vf_01_tensionX_remeshed_increment19000_segmented.dream3D"
}

MY_MOBILITY_RH: List[Tuple[str, Quantity]] = [
    (
        "m_null",
        ureg.Quantity(np.float64(300.0), ureg.meter**4 / (ureg.joule * ureg.second)),
    ),
    ("enthalpy", ureg.Quantity(np.float64(2.0314), ureg.eV)),
    # ("c1", ureg.Quantity(np.float64(0.9))),
    # ("c2", ureg.Quantity(np.float64(5.))),
    # ("c3", ureg.Quantity(np.float64(9.))),
]

MY_TIME_TEMPERATURE: List[Tuple[str, Quantity]] = [
    (
        "time",
        ureg.Quantity(
            np.asarray(
                [
                    0.0,
                    5.8,
                    100.8,
                    103.7,
                    105.0,
                    200.8,
                    205.8,
                    208.7,
                    300.0,
                    305.8,
                    405.8,
                    408.7,
                    600.0,
                ],
                np.float64,
            ),
            ureg.second,
        ),
    ),
    (
        "temperature",
        ureg.Quantity(
            np.asarray(
                [
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                ],
                np.float64,
            ),
            ureg.degC,
        ).to(ureg.kelvin),
    ),
]

MY_GRID_CUBOIDAL: List[Tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([19, 15, 11], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.5), ureg.micrometer)),
]

MY_GRID_DAMASK: List[Tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([128, 52, 64], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.4982421), ureg.micrometer)),
]

MY_SIM_CONTROL: List[Tuple[str, Quantity]] = [
    ("max_x", ureg.Quantity(np.float64(1.0))),
    ("max_time", ureg.Quantity(np.float64(3600.0), ureg.second)),
    ("max_iteration", ureg.Quantity(np.uint32(10000))),
    ("max_delta_x", ureg.Quantity(np.float64(0.1))),
    (
        "x_set",
        ureg.Quantity(
            np.asarray(np.linspace(0.0, stop=1.0, num=101, endpoint=True), np.float64)
        ),
    ),
]

In [ ]:
print("Generating NeXus/HDF5 configuration file for examples/verification")
nxs = ScoreConfig(0)
nxs.deformation()
nxs.nucleation()
nxs.numerics(MY_SIM_CONTROL)
nxs.get_config()
nxs.write_config_hfive_file()
del nxs

In [ ]:
print("Generating NeXus/HDF5 configuration file for examples/damask")
nxs = ScoreConfig(1)
nxs.user(MY_USER)
nxs.sample("Al")
nxs.material(MY_MATERIAL)
nxs.deformation("damask", MY_INIT_DAMASK)
nxs.nucleation("damask")
nxs.mobility("rollett_holm", MY_MOBILITY_RH)
nxs.time_temperature(MY_TIME_TEMPERATURE)
nxs.discretization(MY_GRID_DAMASK)
nxs.numerics(MY_SIM_CONTROL)
nxs.get_config()
nxs.write_config_hfive_file()
del nxs